# Build a Child-Like AI Agent in Python
### A Real-Math, Real-Model Workshop

---

**What this notebook covers:**
- TF-IDF vectorization (the perception layer)
- Cosine similarity (the decision layer)
- Bayesian belief strength updates (the learning layer)
- An incremental Naive Bayes classifier (the real model inside the identity)
- A full `perceive → decide → evaluate → update` loop
- Live demo with healthcare triage examples

**What this is NOT:**
- Reinforcement Learning (no reward signal, no policy, no value function)
- A pretrained model wrapper (we build every component from scratch)
- A memory-database architecture (learning lives *inside* identity, not outside it)

---
## 0. Imports

In [ ]:
from __future__ import annotations

import math
import json
from copy import deepcopy
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple
from pprint import pformat

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity

LINE = "=" * 78
DIVIDER = "-" * 78

print("Imports OK")

---
## 1. The Mathematics of Perception: TF-IDF

Before the agent can learn, it must convert raw text into a vector it can reason over.
We use **TF-IDF** (Term Frequency–Inverse Document Frequency).

### Formula

$$\text{TF}(t, d) = \frac{f_{t,d}}{\sum_{t'} f_{t',d}}$$

$$\text{IDF}(t) = \log\left(\frac{N + 1}{df_t + 1}\right) + 1$$

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

Where:
- $f_{t,d}$ = count of term $t$ in document $d$
- $N$ = total number of documents seen so far
- $df_t$ = number of documents containing term $t$

**Why this matters:** Rare, specific words (e.g. "tachycardia") get high weight. Common words (e.g. "is", "a") get low weight. The agent learns *what matters*, not just *what appears*.

The result is a **sparse vector** in $\mathbb{R}^V$ where $V$ is the vocabulary size.

In [ ]:
# ── Live demo: watch TF-IDF transform two sentences ──────────────────────────

demo_corpus = [
    "patient has fever and cough",
    "patient has chest pain",
    "fever is common in malaria",
]

tfidf_demo = TfidfVectorizer(norm="l2")
matrix = tfidf_demo.fit_transform(demo_corpus).toarray()

vocab = tfidf_demo.get_feature_names_out()
print("Vocabulary:", list(vocab))
print()
for i, sentence in enumerate(demo_corpus):
    print(f"Document {i}: '{sentence}'")
    nonzero = [(vocab[j], round(matrix[i, j], 4)) for j in matrix[i].nonzero()[0]]
    print(f"  TF-IDF weights: {nonzero}")
    print()

---
## 2. The Mathematics of Decision: Cosine Similarity

Once we have vectors, the agent measures how similar a new input is to everything it already knows.

### Formula

$$\cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \cdot \|\mathbf{v}\|}$$

Where:
- $\mathbf{u}$ is the vector of the new input
- $\mathbf{v}$ is the **centroid** of all vectors in a known belief category
- The result is in $[0, 1]$ when vectors are non-negative (TF-IDF weights are always ≥ 0)

**Centroid of category $C$:**

$$\bar{\mathbf{v}}_C = \frac{1}{|C|} \sum_{e \in C} \mathbf{v}_e$$

**Why not Euclidean distance?**  
Cosine similarity is length-invariant — it doesn't matter if one document is longer. This is crucial for short clinical notes where word count varies wildly.

In [ ]:
# ── Live demo: cosine similarity between two phrases ─────────────────────────

phrases = [
    "patient has high fever",
    "child shows elevated temperature",  # semantically similar
    "truck engine failure",              # unrelated
]

tfidf_cos = TfidfVectorizer(norm="l2")
vecs = tfidf_cos.fit_transform(phrases).toarray()

query = vecs[0].reshape(1, -1)
for i, phrase in enumerate(phrases):
    sim = cosine_similarity(query, vecs[i].reshape(1, -1))[0][0]
    print(f"cos(query, '{phrase}') = {sim:.4f}")

---
## 3. The Mathematics of Learning: Bayesian Belief Updates

The agent doesn't just count how many times it has seen a category. It maintains a **Bayesian confidence** over each belief.

### Beta-Binomial Model

Each belief category $C$ has a prior:

$$P(\text{correct} | C) \sim \text{Beta}(\alpha_C, \beta_C)$$

After each interaction:
- Correct prediction or useful new info → $\alpha_C \leftarrow \alpha_C + 1$
- Incorrect or redundant → $\beta_C \leftarrow \beta_C + 1$

**Expected belief strength (posterior mean):**

$$\mathbb{E}[P_C] = \frac{\alpha_C}{\alpha_C + \beta_C}$$

**Why this is better than a counter:**
- It's calibrated: a category seen 1 time with 1 correct prediction ≠ category seen 10 times with 10 correct predictions
- It starts uncertain (flat prior: $\alpha=1, \beta=1$) and sharpens with evidence
- It's inspectable and explainable

In [ ]:
# ── Live demo: watch belief strength evolve ───────────────────────────────────

import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
stages = [
    (1, 1, "Prior: no evidence"),
    (3, 2, "After 4 interactions (3 correct)"),
    (9, 2, "After 10 interactions (9 correct)"),
]

x = np.linspace(0, 1, 200)
for ax, (a, b, title) in zip(axes, stages):
    y = beta_dist.pdf(x, a, b)
    mean = a / (a + b)
    ax.plot(x, y, lw=2, color="steelblue")
    ax.axvline(mean, color="red", linestyle="--", label=f"Mean={mean:.2f}")
    ax.fill_between(x, y, alpha=0.15, color="steelblue")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Belief strength P(correct)")
    ax.legend(fontsize=9)
    ax.set_ylim(0, None)

plt.suptitle("Bayesian Belief Strength Over Time (Beta-Binomial)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("/home/claude/belief_update.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved to belief_update.png")

---
## 4. The Real Model: Incremental Naive Bayes

Inside the identity, we embed a **Complement Naive Bayes** classifier that can be updated incrementally with `partial_fit` — no full retraining required.

### The Math

Given class $C$ and feature vector $\mathbf{x}$, Naive Bayes predicts:

$$\hat{C} = \arg\max_C \left[ \log P(C) + \sum_{i} x_i \cdot \log P(f_i | C) \right]$$

**Complement Naive Bayes** learns from the *complement* of each class (what a class is *not*), which works better for imbalanced, sparse text data:

$$\hat{C} = \arg\min_C \left[ \sum_{i} x_i \cdot \log P(f_i | \bar{C}) \right]$$

**`partial_fit`** means the model updates its sufficient statistics (word counts per class) from new examples — without seeing old data again. This is how the identity evolves from interaction.

### Why Not a Neural Network?

| Property | Naive Bayes (ours) | Neural Net |
|---|---|---|
| Incremental update | ✅ native | ❌ needs full retrain |
| Runs on CPU only | ✅ | ❌ slow |
| Inspectable weights | ✅ | ❌ black box |
| Data needed to start | 1 example | hundreds |
| Works offline | ✅ | ✅ |

---
## 5. The Full Agent

In [ ]:
@dataclass
class Decision:
    prediction: str
    confidence: float     # cosine similarity score [0, 1]
    model_confidence: float  # Naive Bayes posterior probability [0, 1]
    reason: str


@dataclass
class Belief:
    """One category in the identity store, with full Bayesian state."""
    examples: List[str] = field(default_factory=list)
    alpha: float = 1.0   # Beta-Binomial: successes + 1 (Laplace smoothing)
    beta: float = 1.0    # Beta-Binomial: failures + 1

    @property
    def strength(self) -> float:
        """Posterior mean of Beta(alpha, beta) — Bayesian belief strength."""
        return self.alpha / (self.alpha + self.beta)


class ChildLikeAgent:
    """
    A child-like AI agent with a real model embedded in its identity.

    Architecture:
    - Perception  : TF-IDF vectorizer (incremental, fits as corpus grows)
    - Decision    : Cosine similarity against belief centroids
                    + Complement Naive Bayes posterior probability
    - Evaluation  : Usefulness gate (novelty + correction + uncertainty)
    - Learning    : Beta-Binomial belief update + partial_fit on the model

    Nothing here is reinforcement learning:
    - No reward signal
    - No policy gradient
    - No value function
    - No replay buffer
    """

    CONFIDENCE_THRESHOLD = 0.30  # cosine similarity below this → "I don't know"
    MODEL_THRESHOLD = 0.50       # Bayes posterior below this → defer to similarity
    MIN_EXAMPLES_FOR_MODEL = 4   # need at least this many examples before trusting the model

    def __init__(self, name: str = "Milo") -> None:
        # ── Identity store ────────────────────────────────────────────────────
        self.identity: Dict[str, Any] = {
            "name": name,
            "version": 1,
            "beliefs": {},          # category → Belief
            "interaction_count": 0,
            "learning_count": 0,
            "ignored_count": 0,
            "last_update_reason": "Agent initialized.",
        }
        # ── Perception layer ──────────────────────────────────────────────────
        self._vectorizer = TfidfVectorizer(norm="l2", sublinear_tf=True)
        self._corpus: List[str] = []          # all texts seen so far
        self._corpus_labels: List[str] = []   # aligned labels
        self._fitted = False

        # ── Real model (Naive Bayes) ──────────────────────────────────────────
        self._model = ComplementNB(alpha=1.0)  # alpha = Laplace smoothing
        self._label_encoder = LabelEncoder()
        self._model_ready = False

    # ──────────────────────────────────────────────────────────────────────────
    # PERCEPTION
    # ──────────────────────────────────────────────────────────────────────────
    def perceive(self, text: str) -> Dict[str, Any]:
        """
        Transform raw text into a TF-IDF vector.

        If the corpus has been seen before, refit the vectorizer on the
        full corpus so IDF weights are always current.
        """
        normalized = text.strip().lower()
        tokens = normalized.split()

        # Fit/refit the vectorizer on the full known corpus + new text
        fit_corpus = self._corpus + [normalized]
        self._vectorizer.fit(fit_corpus)
        self._fitted = True

        vec = self._vectorizer.transform([normalized]).toarray()[0]

        return {
            "raw_text": text,
            "normalized": normalized,
            "tokens": tokens,
            "vector": vec,          # np.ndarray, shape (V,)
            "vocab_size": len(self._vectorizer.vocabulary_),
        }

    # ──────────────────────────────────────────────────────────────────────────
    # DECISION
    # ──────────────────────────────────────────────────────────────────────────
    def decide(self, perception: Dict[str, Any]) -> Decision:
        """
        Two-stage decision:
        1. Cosine similarity against per-category centroids (always available)
        2. Naive Bayes posterior (available once MIN_EXAMPLES_FOR_MODEL reached)

        Final confidence = weighted blend of both signals when model is ready.
        """
        beliefs = self.identity["beliefs"]
        input_vec = perception["vector"]

        if not beliefs:
            return Decision(
                prediction="I don't know yet.",
                confidence=0.0,
                model_confidence=0.0,
                reason="No beliefs in identity yet.",
            )

        # ── Stage 1: cosine similarity ────────────────────────────────────────
        best_cat: Optional[str] = None
        best_cos = 0.0

        for cat, belief in beliefs.items():
            if not belief.examples:
                continue
            # Reproject stored examples through the current vectorizer
            example_vecs = self._vectorizer.transform(belief.examples).toarray()
            centroid = example_vecs.mean(axis=0)              # mean vector
            norm = np.linalg.norm(centroid)
            if norm == 0:
                continue
            centroid_normed = centroid / norm
            cos = float(np.dot(input_vec, centroid_normed))   # cosine similarity
            # Weight by Bayesian belief strength
            weighted = cos * belief.strength
            if weighted > best_cos:
                best_cos = weighted
                best_cat = cat

        # ── Stage 2: Naive Bayes (when ready) ────────────────────────────────
        model_conf = 0.0
        model_pred = None
        total_examples = sum(len(b.examples) for b in beliefs.values())

        if self._model_ready and total_examples >= self.MIN_EXAMPLES_FOR_MODEL:
            try:
                X = self._vectorizer.transform([perception["normalized"]])
                probs = self._model.predict_proba(X)[0]
                best_idx = int(np.argmax(probs))
                model_conf = float(probs[best_idx])
                model_pred = self._label_encoder.classes_[best_idx]
            except Exception:
                pass  # model not ready yet

        # ── Blend signals ─────────────────────────────────────────────────────
        if model_pred and model_conf >= self.MODEL_THRESHOLD:
            # Trust model more as evidence grows
            blend = 0.4 * best_cos + 0.6 * model_conf
            final_pred = model_pred
            reason = (
                f"Model posterior={model_conf:.3f}, "
                f"cosine={best_cos:.3f}, blend={blend:.3f}"
            )
        else:
            blend = best_cos
            final_pred = best_cat or "I don't know."
            reason = f"Cosine similarity (weighted by belief strength) = {best_cos:.3f}"

        if blend < self.CONFIDENCE_THRESHOLD or best_cat is None:
            return Decision(
                prediction="I don't know.",
                confidence=round(blend, 4),
                model_confidence=round(model_conf, 4),
                reason=reason + " — below threshold.",
            )

        return Decision(
            prediction=final_pred,
            confidence=round(blend, 4),
            model_confidence=round(model_conf, 4),
            reason=reason,
        )

    # ──────────────────────────────────────────────────────────────────────────
    # EVALUATE USEFULNESS
    # ──────────────────────────────────────────────────────────────────────────
    def evaluate_usefulness(
        self,
        perception: Dict[str, Any],
        decision: Decision,
        feedback_label: str,
    ) -> Dict[str, Any]:
        """
        The agent decides whether this feedback should update its identity.

        Rules (NOT reward signals):
        - New category → useful (novelty)
        - New example  → useful (coverage)
        - Was uncertain (confidence < threshold) → useful
        - Was wrong    → useful (correction)
        - Otherwise    → redundant, skip
        """
        beliefs = self.identity["beliefs"]
        known = beliefs.get(feedback_label)

        is_new_category = known is None
        is_new_example = not known or perception["normalized"] not in known.examples
        was_uncertain = decision.confidence < self.CONFIDENCE_THRESHOLD
        was_wrong = (
            decision.prediction not in {feedback_label, "I don't know.", "I don't know yet."}
        )

        useful = is_new_category or is_new_example or was_uncertain or was_wrong

        reasons = []
        if is_new_category: reasons.append("new category")
        if is_new_example:  reasons.append("new example")
        if was_uncertain:   reasons.append("agent was uncertain")
        if was_wrong:       reasons.append("agent was wrong")
        if not reasons:     reasons.append("redundant")

        return {"useful": useful, "reason": ", ".join(reasons)}

    # ──────────────────────────────────────────────────────────────────────────
    # LEARN (identity update)
    # ──────────────────────────────────────────────────────────────────────────
    def learn(
        self,
        perception: Dict[str, Any],
        decision: Decision,
        feedback_label: str,
        usefulness: Dict[str, Any],
    ) -> bool:
        """
        Update the identity when feedback is useful:

        1. Add example to belief store
        2. Bayesian belief update: alpha += 1 if correct/new, beta += 1 if wrong
        3. partial_fit the Naive Bayes model on the new example
        4. Increment version counter
        """
        self.identity["interaction_count"] += 1

        if not usefulness["useful"]:
            self.identity["ignored_count"] += 1
            self.identity["last_update_reason"] = f"Skipped: {usefulness['reason']}."
            # Bayesian penalty: beta += 1 for the predicted category (if any)
            if decision.prediction in self.identity["beliefs"]:
                self.identity["beliefs"][decision.prediction].beta += 1
            return False

        beliefs = self.identity["beliefs"]
        text = perception["normalized"]

        # ── 1. Update belief store ─────────────────────────────────────────
        if feedback_label not in beliefs:
            beliefs[feedback_label] = Belief()
        belief = beliefs[feedback_label]

        if text not in belief.examples:
            belief.examples.append(text)

        # ── 2. Bayesian update ─────────────────────────────────────────────
        was_correct = decision.prediction == feedback_label
        if was_correct:
            belief.alpha += 1.0   # reinforcement: right category confirmed
        else:
            belief.alpha += 0.5   # learning something new, partial credit
            belief.beta += 0.5    # some uncertainty acknowledged

        # ── 3. Update corpus and partial_fit the model ─────────────────────
        self._corpus.append(text)
        self._corpus_labels.append(feedback_label)

        all_classes = list(set(self._corpus_labels))
        if len(all_classes) >= 2:
            # Refit vectorizer on full corpus (IDF update) then retrain
            # model from scratch on the full corpus so feature dimensions
            # are always consistent after vocabulary grows.
            self._vectorizer.fit(self._corpus)
            X_all = self._vectorizer.transform(self._corpus)
            self._label_encoder.fit(all_classes)
            y_all = self._label_encoder.transform(self._corpus_labels)
            self._model = ComplementNB(alpha=1.0)
            self._model.partial_fit(
                X_all, y_all,
                classes=self._label_encoder.transform(all_classes)
            )
            self._model_ready = True

        # ── 4. Update identity metadata ────────────────────────────────────
        self.identity["learning_count"] += 1
        self.identity["version"] += 1
        self.identity["last_update_reason"] = (
            f"Learned '{feedback_label}': {usefulness['reason']}. "
            f"Belief strength={belief.strength:.3f} "
            f"(α={belief.alpha:.1f}, β={belief.beta:.1f})."
        )
        return True

print("Agent class defined.")

---
## 6. Healthcare Dataset

Real-world scenario: a low-resource clinic uses the agent to help triage patient complaints when no internet and no GPU are available.

In [ ]:
HEALTHCARE_INTERACTIONS = [
    # ── Respiratory ──────────────────────────────────────────────────────────
    {"input": "patient has persistent cough and shortness of breath",    "label": "respiratory"},
    {"input": "child shows wheezing and labored breathing",              "label": "respiratory"},
    {"input": "adult reports chest tightness and rapid breathing",       "label": "respiratory"},
    {"input": "elderly patient with chronic cough and mucus production", "label": "respiratory"},
    # ── Fever / Infection ─────────────────────────────────────────────────────
    {"input": "patient presents with high fever and chills",            "label": "fever_infection"},
    {"input": "child has 39 degree temperature and body aches",         "label": "fever_infection"},
    {"input": "adult reports sweating and elevated temperature",        "label": "fever_infection"},
    {"input": "patient shows signs of malaria high fever and fatigue",  "label": "fever_infection"},
    # ── Cardiac ───────────────────────────────────────────────────────────────
    {"input": "patient has chest pain radiating to the left arm",       "label": "cardiac"},
    {"input": "adult reports palpitations and irregular heartbeat",     "label": "cardiac"},
    {"input": "elderly shows tachycardia and dizziness on exertion",    "label": "cardiac"},
    # ── Gastrointestinal ──────────────────────────────────────────────────────
    {"input": "patient reports severe abdominal pain and vomiting",     "label": "gastrointestinal"},
    {"input": "child has diarrhea and stomach cramps since yesterday",  "label": "gastrointestinal"},
    {"input": "adult presents with nausea and loss of appetite",        "label": "gastrointestinal"},
    # ── Neurological ──────────────────────────────────────────────────────────
    {"input": "patient has severe headache and light sensitivity",      "label": "neurological"},
    {"input": "child shows confusion and seizure activity",             "label": "neurological"},
    {"input": "adult reports numbness in left hand and slurred speech", "label": "neurological"},
    # ── Unseen test queries (not in training) ─────────────────────────────────
    {"input": "patient breathes with difficulty after running",         "label": "respiratory"},
    {"input": "child shows very fast heart rate during sleep",          "label": "cardiac"},
    {"input": "patient has high temperature and joint pain",            "label": "fever_infection"},
]

print(f"{len(HEALTHCARE_INTERACTIONS)} interactions ready.")

---
## 7. Run the Full Learning Loop

In [ ]:
def run_agent(interactions: List[Dict[str, str]], verbose: bool = True) -> ChildLikeAgent:
    agent = ChildLikeAgent(name="Milo-Health")

    print(LINE)
    print("CHILD-LIKE AI AGENT — HEALTHCARE TRIAGE DEMO")
    print("Architecture: TF-IDF + Cosine Similarity + Complement Naive Bayes")
    print("Learning:     Beta-Binomial Bayesian Updates + partial_fit")
    print(LINE)

    results = []

    for i, item in enumerate(interactions, 1):
        before = deepcopy(agent.identity)

        perception  = agent.perceive(item["input"])
        decision    = agent.decide(perception)
        usefulness  = agent.evaluate_usefulness(perception, decision, item["label"])
        learned     = agent.learn(perception, decision, item["label"], usefulness)

        correct = decision.prediction == item["label"]
        results.append(correct)

        if verbose:
            print(DIVIDER)
            print(f"[{i:02d}] INPUT     : {item['input']}")
            print(f"     PREDICTION: {decision.prediction}")
            print(f"     TRUE LABEL: {item['label']}")
            print(f"     CONFIDENCE: cosine={decision.confidence:.4f}  model={decision.model_confidence:.4f}")
            print(f"     CORRECT?  : {'✓' if correct else '✗'}")
            print(f"     USEFUL?   : {usefulness['useful']}  ({usefulness['reason']})")
            print(f"     LEARNED?  : {learned}")
            print(f"     UPDATE    : {agent.identity['last_update_reason']}")

            # Show current belief strengths
            if agent.identity["beliefs"]:
                strengths = {
                    cat: f"{b.strength:.3f} (α={b.alpha:.1f},β={b.beta:.1f})"
                    for cat, b in agent.identity["beliefs"].items()
                }
                print(f"     BELIEFS   : {json.dumps(strengths, indent=18)[1:-1].strip()}")

    print(LINE)
    print("FINAL SUMMARY")
    print(LINE)
    print(f"Interactions : {agent.identity['interaction_count']}")
    print(f"Learned      : {agent.identity['learning_count']}")
    print(f"Ignored      : {agent.identity['ignored_count']}")
    print(f"Identity ver : {agent.identity['version']}")
    accuracy = sum(results) / len(results)
    print(f"Accuracy     : {accuracy:.1%}  ({sum(results)}/{len(results)} correct)")
    print("Categories   :", ", ".join(agent.identity["beliefs"].keys()))

    return agent


agent = run_agent(HEALTHCARE_INTERACTIONS, verbose=True)

---
## 8. Visualize: Confidence Growth Over Time

In [ ]:
def plot_confidence_evolution(interactions: List[Dict[str, str]]) -> None:
    agent = ChildLikeAgent(name="Milo-Plot")
    confidences = []
    model_confidences = []
    correct_flags = []

    for item in interactions:
        perception = agent.perceive(item["input"])
        decision = agent.decide(perception)
        usefulness = agent.evaluate_usefulness(perception, decision, item["label"])
        agent.learn(perception, decision, item["label"], usefulness)

        confidences.append(decision.confidence)
        model_confidences.append(decision.model_confidence)
        correct_flags.append(decision.prediction == item["label"])

    x = list(range(1, len(interactions) + 1))
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Confidence over time
    ax1.plot(x, confidences, label="Cosine confidence", color="steelblue", lw=2)
    ax1.plot(x, model_confidences, label="Model (Naive Bayes) posterior", color="darkorange", lw=2, linestyle="--")
    ax1.axhline(ChildLikeAgent.CONFIDENCE_THRESHOLD, color="red", linestyle=":", label="Threshold")
    for i, (c, ok) in enumerate(zip(confidences, correct_flags), 1):
        color = "green" if ok else "red"
        ax1.scatter(i, c, color=color, zorder=5, s=50)
    ax1.set_ylabel("Confidence")
    ax1.set_title("Agent Confidence Over Interactions (green = correct, red = wrong/unknown)")
    ax1.legend()
    ax1.set_ylim(-0.05, 1.05)

    # Belief strength over time (posterior means)
    agent2 = ChildLikeAgent(name="Milo-Plot2")
    history = defaultdict(list)
    for item in interactions:
        perception = agent2.perceive(item["input"])
        decision = agent2.decide(perception)
        usefulness = agent2.evaluate_usefulness(perception, decision, item["label"])
        agent2.learn(perception, decision, item["label"], usefulness)
        for cat, b in agent2.identity["beliefs"].items():
            history[cat].append(b.strength)
        # Pad categories not yet seen
        step = max(len(v) for v in history.values())
        for cat in history:
            while len(history[cat]) < step:
                history[cat].insert(0, 0.5)  # prior mean

    colors = plt.cm.tab10(np.linspace(0, 1, len(history)))
    for (cat, strengths), color in zip(history.items(), colors):
        ax2.plot(range(1, len(strengths) + 1), strengths, label=cat, color=color, lw=2)

    ax2.axhline(0.5, color="gray", linestyle=":", label="Prior mean (uninformed)")
    ax2.set_xlabel("Interaction #")
    ax2.set_ylabel("Bayesian belief strength E[P]")
    ax2.set_title("Belief Strength per Category Over Time (Beta posterior mean)")
    ax2.legend(fontsize=8, ncol=2)

    plt.tight_layout()
    plt.savefig("/home/claude/confidence_evolution.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved to confidence_evolution.png")


plot_confidence_evolution(HEALTHCARE_INTERACTIONS)

---
## 9. Visualize: TF-IDF Feature Space (PCA)

Let's see how the agent's perception space organizes the healthcare examples.

In [ ]:
from sklearn.decomposition import PCA

def plot_feature_space(agent: ChildLikeAgent) -> None:
    if not agent._corpus:
        print("No corpus yet.")
        return

    vecs = agent._vectorizer.transform(agent._corpus).toarray()
    labels = agent._corpus_labels

    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(vecs)

    unique_labels = sorted(set(labels))
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    label_color = {l: c for l, c in zip(unique_labels, colors)}

    fig, ax = plt.subplots(figsize=(10, 7))
    for label in unique_labels:
        idx = [i for i, l in enumerate(labels) if l == label]
        ax.scatter(
            coords[idx, 0], coords[idx, 1],
            label=label, color=label_color[label], s=100, alpha=0.8
        )
        # Draw centroid
        cx, cy = coords[idx, 0].mean(), coords[idx, 1].mean()
        ax.scatter(cx, cy, color=label_color[label], s=300, marker="*", edgecolors="black", zorder=5)

    variance = pca.explained_variance_ratio_
    ax.set_xlabel(f"PC1 ({variance[0]:.1%} variance)")
    ax.set_ylabel(f"PC2 ({variance[1]:.1%} variance)")
    ax.set_title("TF-IDF Feature Space (PCA) — Stars = Category Centroids")
    ax.legend()
    plt.tight_layout()
    plt.savefig("/home/claude/feature_space.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"PCA explained variance: PC1={variance[0]:.1%}, PC2={variance[1]:.1%}")


plot_feature_space(agent)

---
## 10. Live Interactive Demo

Type your own clinical note and see the agent respond in real time.

In [ ]:
def live_demo(agent: ChildLikeAgent) -> None:
    """
    Interactive loop for workshop audience participation.
    Type a patient symptom description — the agent classifies it.
    Then type the correct label to teach the agent.
    Type 'quit' to exit.
    """
    print(LINE)
    print("LIVE INTERACTIVE DEMO")
    print("Type a patient symptom description. The agent will classify it.")
    print("Then optionally provide the correct label to teach the agent.")
    print(f"Known categories: {', '.join(agent.identity['beliefs'].keys())}")
    print("Type 'quit' to exit.")
    print(LINE)

    while True:
        text = input("\nSymptom description: ").strip()
        if text.lower() == "quit":
            break
        if not text:
            continue

        perception = agent.perceive(text)
        decision = agent.decide(perception)

        print(f"  → Prediction : {decision.prediction}")
        print(f"  → Confidence : cosine={decision.confidence:.4f}  model={decision.model_confidence:.4f}")
        print(f"  → Reason     : {decision.reason}")

        label = input("  Correct label (Enter to skip): ").strip()
        if label:
            usefulness = agent.evaluate_usefulness(perception, decision, label)
            learned = agent.learn(perception, decision, label, usefulness)
            print(f"  → Useful: {usefulness['useful']} ({usefulness['reason']})")
            print(f"  → Learned: {learned}")
            if learned:
                b = agent.identity["beliefs"][label]
                print(f"  → Belief strength: {b.strength:.3f} (α={b.alpha:.1f}, β={b.beta:.1f})")


# Uncomment the line below to run live in the workshop:
# live_demo(agent)

---
## 11. Why This Is Not Reinforcement Learning — Side-by-Side

| | Reinforcement Learning | This Agent |
|---|---|---|
| **Learning signal** | Scalar reward $r_t$ | Human label (categorical) |
| **Objective** | Maximize $\sum \gamma^t r_t$ | No objective function |
| **What updates** | Policy $\pi_\theta$ or $Q(s,a)$ | Identity beliefs + model weights |
| **Math** | Policy gradient / Bellman eq. | Bayesian update + partial_fit |
| **Exploration** | $\epsilon$-greedy, UCB | Uncertainty → "I don't know" |
| **Retraining needed** | Often | Never (incremental) |
| **Interpretable?** | Rarely | Always (inspect beliefs dict) |

---
## 12. Workshop Checkpoints

After running this notebook, you should be able to answer:

1. **What does TF-IDF actually compute?** Walk through the formula with a clinical example.
2. **Why cosine similarity, not Euclidean distance?** What property makes it better for variable-length text?
3. **What does `partial_fit` do differently from `fit`?** Why does this matter for a self-evolving agent?
4. **What is β in the Beta-Binomial model?** When does it increase?
5. **Why does the agent say "I don't know"?** What threshold controls this, and how is it calibrated?
6. **Could you replace the Naive Bayes model with a different sklearn estimator?** What interface requirement would it need to satisfy?